# CNN to classify handwritten digits

The MNIST dataset is a classic model dataset, conveniently built into Pytorch. (Some information e.g. here: https://www.geeksforgeeks.org/machine-learning/mnist-dataset/, https://en.wikipedia.org/wiki/MNIST_database). It contains a training set of 60000 and a test set of 10000 handwritten digits

It is very clean and works unusually well, due to...
- centered digits
- low noise
- simple shapes

A very simple CNN is therefore enough.

Concerning the data: Each image (=each sample) was normalised to fit a 28x28 pixel bounding box.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Load the data
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="data", train=True,
                               download=True, transform=transform)

test_dataset = datasets.MNIST(root="data", train=False,
                              download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=1000)

Show some datapoints

In [ ]:
import matplotlib.pyplot as plt
images, labels = next(iter(train_loader))
print(images.shape)   # torch.Size([32, 1, 28, 28])
print(labels.shape)   # torch.Size([32])

fig, axes = plt.subplots(1, 8, figsize=(12, 2))

for i in range(8):
    ax = axes[i]
    ax.imshow(images[i].squeeze(), cmap="gray")
    ax.set_title(str(labels[i].item()))
    ax.axis("off")

plt.show()


Build a CNN! Decide how many convolutional layers you want to have - set the parameters for each. Then define the linear layers for classification (define width according to the channels from the last convolutional layer).

In [ ]:
class MNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        # 28x28 pixel of input pictures, one channel (grayscale)
        self.conv = nn.Sequential(
            nn.Conv2d(
                in_channels=, 
                out_channels=, 
                kernel_size=, 
                padding=1),  # output size: 28x28 (same as input due to padding!)
            nn.ReLU(),
            nn.MaxPool2d(2),                 
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(,)
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

Initialise model, set optimiser and loss function.

In [ ]:
model = MNIST_CNN()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

Use the defined training and test functions:

In [ ]:
def train():
    model.train()
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

In [ ]:
def test():
    model.eval()
    correct = 0

    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()

    return correct / len(test_dataset)

Train and evaluate the model per epoch (don't overdo it with the epochs).

In [ ]:
for epoch in range(1):
    train()
    acc = test()
    print(f"Epoch {epoch+1}: Test Accuracy = {acc:.4f}")

# Recognise your own handwriting?

Use the model to classify your own handwritten images. For this though, you have to get the image into exactly the same input format as in the dataset.

Write some digits - either on paper (high contrast!) or virtually - make sure to save the image as PNG. In order to get the preprocessing done, use the following snippet (colour inversion implemented, normalisation parameters from MNIST dataset).

In [ ]:
from PIL import Image
import torchvision.transforms as transforms
import torch

# Use the SAME transform that MNIST used (important!)
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: 1 - x),
    transforms.Normalize((0.1307,), (0.3081,))
])

def load_own_digit(path):
    img = Image.open(path).convert("L")  # "L" converts to grayscale
    img = transform(img)
    img = img.unsqueeze(0)  # add batch dimension
    return img

Use the following code to see what the model reads in your handwriting!

In [ ]:
model.eval()

img = load_own_digit("my_digit.png")  # path to your digit image

# visualise your picture :)
plt.imshow(img.squeeze().numpy(), cmap="gray")
plt.title("Processed Image")
plt.show()

output = model(img)

prediction = output.argmax(1).item()
print("Predicted digit:", prediction)